# Thesis — 3-Label Stage 2 with Optuna Hyperparameter Tuning

This notebook adds **Optuna hyperparameter tuning** on top of the base experiment.
The EDA, preprocessing, train/validation/test splits, tokenization, and metrics are
**identical** to the base notebook. The only additions are:

- an Optuna `study` per stage that searches over `learning_rate`, `num_train_epochs`,
  `per_device_train_batch_size`, and `weight_decay`;
- each trial trains on the **train** split and is scored on the **validation** split
  (never the test set), so tuning never sees the held-out test data;
- Stage 1 optimizes toxic-class **F1**; Stage 2 optimizes **macro F1**;
- after the search, the best hyperparameters are used to train a final model, which is
  then evaluated once on the untouched **test** set (with decision-threshold tuning
  done on validation).

`N_TRIALS` is kept small so the notebook finishes in reasonable time on Kaggle; raise it
for a more thorough search.

In [1]:
# Install Optuna if it isn't already available in the environment.
# On Kaggle this is usually a no-op after the first run.
!pip install -q optuna

In [2]:
import pandas as pd
import numpy as np
import tqdm

from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from huggingface_hub import login

import torch
from torch import nn
import torch.nn.functional as F
from sklearn.utils.class_weight import compute_class_weight

import optuna

import ast
import re

# Silence transformers' informational/warning logs (e.g. legacy LayerNorm
# gamma/beta key-name notices when Trainer reloads the best checkpoint).
# Doesn't affect training/evaluation, only console noise.
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

In [3]:
df = pd.read_csv("/kaggle/input/datasets/evelynuntariady/thesis/data_thesis_majority_vote.csv")

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26379 entries, 0 to 26378
Data columns (total 15 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   Unnamed: 0                     26379 non-null  int64 
 1   text_id                        26379 non-null  object
 2   annotators_id                  26379 non-null  object
 3   text                           26379 non-null  object
 4   topic                          26379 non-null  object
 5   is_noise_or_spam_text          26379 non-null  int64 
 6   toxicity                       26379 non-null  int64 
 7   polarized                      26379 non-null  int64 
 8   profanity_obscenity            26379 non-null  int64 
 9   threat_incitement_to_violence  26379 non-null  int64 
 10  insults                        26379 non-null  int64 
 11  identity_attack                26379 non-null  int64 
 12  sexually_explicit              26379 non-null  int64 
 13  c

In [5]:
df.head()

,Unnamed: 0,text_id,annotators_id,text,topic,is_noise_or_spam_text,toxicity,polarized,profanity_obscenity,threat_incitement_to_violence,insults,identity_attack,sexually_explicit,cleaned_text,processed_text
0,0,2026-01-01 00:00:00,"['7', '15']",Wajah Jurnalis Dibalut Perban Saat Melaporkan ...,UNKNOWN,0,0,0,0,0,0,0,0,Wajah jurnalis dibalut perban saat melaporkan ...,wajah jurnalis dibalut perban saat melaporkan ...
1,1,2026-01-10 00:00:00,"['7', '15']",Elektabilitas Paslon 02 sebentar lagi terjun b...,Disabilitas,0,0,1,0,0,0,0,0,Elektabilitas pasangan calon 02 sebentar lagi ...,elektabilitas pasangan calon sebentar lagi ter...
2,2,1-1001,"['10', '18']","Kristen, Hindu, Islam dapat perlakuan istimewa...",Kristen,0,0,1,0,0,0,0,0,"Kristen, Hindu, Islam dapat perlakuan istimewa...","kristen, hindu, islam dapat perlakuan istimewa..."
3,3,1-1004,"['10', '18']","Masyarakat Kristen, Hindu, Budha, dan etnis Ti...","Kristen, Tionghoa",0,0,0,0,0,0,0,0,"Masyarakat Kristen, Hindu, Buddha, dan etnis T...","masyarakat kristen, hindu, buddha, dan etnis t..."
4,4,1-1005,"['10', '18']",Cina Mengirimkan Satelit Penginderaan Jauh Bar...,Tionghoa,0,0,0,0,0,0,0,0,Cina mengirimkan satelit penginderaan jauh bar...,cina mengirimkan satelit penginderaan jauh bar...


In [6]:
def transform_datetime_id(text_id_str):
    # Regex to match the datetime pattern like 'YYYY-MM-DD HH:MM:SS'
    match = re.match(r'(\d+)-(\d{2})-(\d{2}) \d{2}:\d{2}:\d{2}', str(text_id_str))
    if match:
        year_part = match.group(1)
        month_part = int(match.group(2))
        day_part = int(match.group(3))

        if year_part == '2026':
            # For year 2026, format as month-day (e.g. '1-1')
            return f"{month_part}-{day_part}"
        else:
            # For other years, format as month-original_year (e.g. '2-8783')
            return f"{month_part}-{year_part}"
    else:
        return text_id_str  # Return original if pattern doesn't match

# Apply this transformation to the 'text_id' column of your DataFrame
df['text_id'] = df['text_id'].apply(transform_datetime_id)

In [7]:
cols = [
'is_noise_or_spam_text',
'toxicity',
'polarized',
'profanity_obscenity',
'threat_incitement_to_violence',
'insults',
'identity_attack',
'sexually_explicit'
]

for col in cols:
    df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x,str) else x)

In [8]:
df_main = df.copy()

In [9]:
df_main = df_main[df_main['is_noise_or_spam_text'] == 0]

In [10]:
df_main.isnull().sum()

Unnamed: 0                       0
text_id                          0
annotators_id                    0
text                             0
topic                            0
is_noise_or_spam_text            0
toxicity                         0
polarized                        0
profanity_obscenity              0
threat_incitement_to_violence    0
insults                          0
identity_attack                  0
sexually_explicit                0
cleaned_text                     0
processed_text                   0
dtype: int64

In [11]:
slang_dict = {
    "gk": "tidak",
    "ga": "tidak",
    "nggak": "tidak",
    "yg": "yang",
    "lu": "kamu",
    "loe": "kamu",
    "gw": "saya",
    "gue": "saya",
    "bgt": "banget",
    "dgn": "dengan",
    "krn": "karena",
    "aja": "saja",
    "trs": "terus"
}

In [12]:
import re

def normalize_slang(text, slang_dict):
    words = text.split()
    return " ".join([slang_dict.get(w.lower(), w) for w in words])

def preprocess_text(text, slang_dict, lowercase=True):
    if not isinstance(text, str):
        return ""

    # 1. remove username (@user)
    text = re.sub(r'@\w+', '', text)

    # 2. remove URL
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # 3. remove numbers
    text = re.sub(r'\d+', '', text)

    # 4. remove unwanted symbols (keep .,!?)
    text = re.sub(r'[^\w\s.,!?]', '', text)

    # 5. normalize slang
    text = normalize_slang(text, slang_dict)

    # 6. normalize repeated chars (contoh: anjiiing -> anjiing)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # 7. lowercase (optional)
    if lowercase:
        text = text.lower()

    # 8. remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [13]:
from tqdm import tqdm
tqdm.pandas()

df_main["processed_text"] = (
    df_main["cleaned_text"]
    .fillna("")
    .astype(str)
    .progress_apply(
        lambda x: preprocess_text(x, slang_dict)
    )
)

100%|██████████| 26379/26379 [00:01<00:00, 19995.23it/s]


In [14]:
df_main['toxicity'].value_counts()

toxicity
0    23980
1     2399
Name: count, dtype: int64

In [15]:
df_main['profanity_obscenity'].value_counts()

profanity_obscenity
0    26018
1      361
Name: count, dtype: int64

In [16]:
df_main['insults'].value_counts()

insults
0    25315
1     1064
Name: count, dtype: int64

In [17]:
df_main['identity_attack'].value_counts()

identity_attack
0    25365
1     1014
Name: count, dtype: int64

In [18]:
KEPT_LABEL_COLS = ["insults", "identity_attack", "profanity_obscenity"]
DROPPED_LABEL_COLS = ["sexually_explicit", "threat_incitement_to_violence"]

# Drop the minority label columns entirely, and -- before ANY train/val/test split,
# Stage 1's included -- drop only the rows whose positive label(s) came SPECIFICALLY
# from those dropped columns (toxicity=1, sexually_explicit and/or
# threat_incitement_to_violence = 1, all 3 kept columns = 0). This is deliberately
# narrower than "kept columns sum to 0": a toxic row can also be all-zero across every
# subtype because it was never actually annotated with one (annotation error), and that
# case has nothing to do with removing these two columns, so it must NOT be dropped here.
only_had_dropped_labels = (
    (df_main["toxicity"] == 1)
    & (df_main[DROPPED_LABEL_COLS].sum(axis=1) > 0)
    & (df_main[KEPT_LABEL_COLS].sum(axis=1) == 0)
)

print("df_main rows before dropping minority-only toxic rows:", len(df_main))
df_main = df_main[~only_had_dropped_labels].copy()
df_main = df_main.drop(columns=DROPPED_LABEL_COLS)
print("df_main rows after dropping minority-only toxic rows:", len(df_main))

df_main rows before dropping minority-only toxic rows: 26379
df_main rows after dropping minority-only toxic rows: 26262


# Stage 1 — Binary Toxicity (Optuna)

In [ ]:
from huggingface_hub import login

login("")

In [20]:
x = df_main['processed_text']
y = df_main['toxicity']

In [21]:
# Split into train / validation / test (70% / 20% / 10%), stratified. Validation drives
# Optuna scoring + checkpoint selection + threshold tuning; test stays untouched until
# the final one-shot evaluation.
x_train_full, x_test, y_train_full, y_test = train_test_split(
    x, y, test_size=0.2, stratify=y, random_state=42
)
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full, test_size=0.125, stratify=y_train_full, random_state=42
)

print("Stage 1 split sizes:", {"train": len(x_train), "val": len(x_val), "test": len(x_test)})

Stage 1 split sizes: {'train': 18382, 'val': 2627, 'test': 5253}


In [22]:
train_df = pd.DataFrame({"processed_text": x_train, "label": y_train})
val_df = pd.DataFrame({"processed_text": x_val, "label": y_val})
test_df = pd.DataFrame({"processed_text": x_test, "label": y_test})

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test": Dataset.from_pandas(test_df)
})

In [23]:
from transformers import (
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    AutoModelForSequenceClassification,
    EarlyStoppingCallback,
)
tokenizer = AutoTokenizer.from_pretrained("indolem/indobertweet-base-uncased")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [24]:
def tokenize(batch):
    return tokenizer(
        batch["processed_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [25]:
tokenized_datasets_1 = dataset.map(tokenize, batched=True)

tokenized_datasets_1 = tokenized_datasets_1.remove_columns(["processed_text"])
tokenized_datasets_1 = tokenized_datasets_1.rename_column("label", "labels")
tokenized_datasets_1.set_format("torch")

train_dataset_1 = tokenized_datasets_1["train"]
val_dataset_1 = tokenized_datasets_1["validation"]
test_dataset_1 = tokenized_datasets_1["test"]

Map:   0%|          | 0/18382 [00:00<?, ? examples/s]

Map:   0%|          | 0/2627 [00:00<?, ? examples/s]

Map:   0%|          | 0/5253 [00:00<?, ? examples/s]

In [26]:
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, auc, roc_curve, precision_recall_curve
)

def compute_metrics_s1(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
        "recall": recall_score(labels, preds),
        "precision": precision_score(labels, preds),
        "roc_auc": auc(roc_curve(labels, preds)[0], roc_curve(labels, preds)[1])
    }

## Optuna search — Stage 1

In [27]:
N_TRIALS = 10  # raise for a more thorough search

def model_init_s1():
    return AutoModelForSequenceClassification.from_pretrained(
        "indolem/indobertweet-base-uncased",
        num_labels=2
    )

def objective_s1(trial):
    import shutil, os
    lr = trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True)
    epochs = trial.suggest_int("num_train_epochs", 3, 6)
    batch_size = trial.suggest_categorical("per_device_train_batch_size", [8, 16])
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.1)

    args = TrainingArguments(
        output_dir=f"./optuna_s1/trial_{trial.number}",
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=16,
        learning_rate=lr,
        weight_decay=weight_decay,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        save_only_model=True,      # don't persist optimizer state during search (saves disk)
        save_total_limit=1,        # keep only the best checkpoint per trial
        metric_for_best_model="f1",       # toxic-class F1
        greater_is_better=True,
        remove_unused_columns=False,
        logging_steps=50,
        disable_tqdm=True,
        report_to=[],
    )

    trainer = Trainer(
        model_init=model_init_s1,
        args=args,
        train_dataset=train_dataset_1,
        eval_dataset=val_dataset_1,
        compute_metrics=compute_metrics_s1,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    try:
        trainer.train()
        metrics = trainer.evaluate()
        return metrics["eval_f1"]
    finally:
        # Free disk between trials: Kaggle's working dir fills up fast, and a
        # half-written checkpoint is what triggers the torch.save iostream error.
        shutil.rmtree(f"./optuna_s1/trial_{trial.number}", ignore_errors=True)

study_s1 = optuna.create_study(direction="maximize", study_name="stage1_3label")
study_s1.optimize(objective_s1, n_trials=N_TRIALS)

print("Best Stage 1 params:", study_s1.best_params)
print("Best Stage 1 validation F1:", study_s1.best_value)

[I 2026-09-23 07:20:53,804] A new study created in memory with name: stage1_3label


pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

{'loss': '0.3569', 'grad_norm': '5.724', 'learning_rate': '2.077e-05', 'epoch': '1.043'}
{'loss': '0.3071', 'grad_norm': '7.881', 'learning_rate': '2.041e-05', 'epoch': '1.13'}
{'loss': '0.3022', 'grad_norm': '4.432', 'learning_rate': '2.004e-05', 'epoch': '1.217'}
{'loss': '0.3894', 'grad_norm': '4.631', 'learning_rate': '1.968e-05', 'epoch': '1.304'}
{'loss': '0.3123', 'grad_norm': '4.886', 'learning_rate': '1.931e-05', 'epoch': '1.391'}
{'loss': '0.3367', 'grad_norm': '7.924', 'learning_rate': '1.895e-05', 'epoch': '1.478'}
{'loss': '0.3544', 'grad_norm': '6.412', 'learning_rate': '1.859e-05', 'epoch': '1.565'}
{'loss': '0.3169', 'grad_norm': '11.18', 'learning_rate': '1.822e-05', 'epoch': '1.652'}
{'loss': '0.3469', 'grad_norm': '18.89', 'learning_rate': '1.786e-05', 'epoch': '1.739'}
{'loss': '0.3868', 'grad_norm': '4.38', 'learning_rate': '1.749e-05', 'epoch': '1.826'}
{'loss': '0.3808', 'grad_norm': '10.14', 'learning_rate': '1.713e-05', 'epoch': '1.913'}
{'loss': '0.2575', 'gra

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '788.6', 'train_samples_per_second': '139.9', 'train_steps_per_second': '4.375', 'train_loss': '0.3435', 'epoch': '3'}


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
[I 2026-09-23 10:27:25,656] Trial 9 finished with value: 0.4873096446700508 and parameters: {'learning_rate': 2.5134827896494853e-05, 'num_train_epochs': 6, 'per_device_train_batch_size': 16, 'weight_decay': 0.016132206554623975}. Best is trial 9 with value: 0.4873096446700508.


{'eval_loss': '0.4081', 'eval_accuracy': '0.9231', 'eval_f1': '0.4873', 'eval_recall': '0.4211', 'eval_precision': '0.5783', 'eval_roc_auc': '0.6959', 'eval_runtime': '12.39', 'eval_samples_per_second': '212', 'eval_steps_per_second': '6.699', 'epoch': '3'}
Best Stage 1 params: {'learning_rate': 2.5134827896494853e-05, 'num_train_epochs': 6, 'per_device_train_batch_size': 16, 'weight_decay': 0.016132206554623975}
Best Stage 1 validation F1: 0.4873096446700508


## Train final Stage 1 model with best hyperparameters

In [28]:
best_s1 = study_s1.best_params

args_s1 = TrainingArguments(
    output_dir="./best_s1",
    num_train_epochs=best_s1["num_train_epochs"],
    per_device_train_batch_size=best_s1["per_device_train_batch_size"],
    per_device_eval_batch_size=16,
    learning_rate=best_s1["learning_rate"],
    weight_decay=best_s1["weight_decay"],
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    remove_unused_columns=False,
    logging_steps=50,
    disable_tqdm=False,
    report_to=[],
)

model_s1 = model_init_s1()
trainer_s1 = Trainer(
    model=model_s1,
    args=args_s1,
    train_dataset=train_dataset_1,
    eval_dataset=val_dataset_1,
    compute_metrics=compute_metrics_s1,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
trainer_s1.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1,Recall,Precision,Roc Auc
1,0.418442,0.408193,0.918538,0.470297,0.416667,0.539773,0.691451
2,0.348209,0.401632,0.921203,0.456693,0.381579,0.568627,0.677034
3,0.263897,0.554792,0.917016,0.480952,0.442982,0.526042,0.702525
4,0.166372,0.727456,0.914351,0.383562,0.307018,0.510949,0.639545
5,0.118886,0.981892,0.907499,0.375321,0.320175,0.453416,0.641747


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2875, training_loss=0.25777953280573307, metrics={'train_runtime': 1326.9643, 'train_samples_per_second': 83.116, 'train_steps_per_second': 2.6, 'total_flos': 6045634274534400.0, 'train_loss': 0.25777953280573307, 'epoch': 5.0})

In [29]:
# Tune the Stage 1 decision threshold on VALIDATION (never test), reusing the trained
# checkpoint. Sweep the toxic-class probability vs a precision-recall curve, pick the
# threshold maximizing F1.
pred1_val = trainer_s1.predict(val_dataset_1)
probs1_val = torch.softmax(torch.tensor(pred1_val.predictions), dim=-1)[:, 1].numpy()

precision_curve_s1, recall_curve_s1, thresholds_s1 = precision_recall_curve(y_val, probs1_val)
f1_curve_s1 = 2 * precision_curve_s1 * recall_curve_s1 / (precision_curve_s1 + recall_curve_s1 + 1e-12)
best_idx_s1 = np.argmax(f1_curve_s1[:-1])
best_threshold_s1 = thresholds_s1[best_idx_s1]

print(f"Stage 1 best threshold (max F1 on validation): {best_threshold_s1:.4f}")
print(f"  validation precision={precision_curve_s1[best_idx_s1]:.4f} "
      f"recall={recall_curve_s1[best_idx_s1]:.4f} f1={f1_curve_s1[best_idx_s1]:.4f}")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Stage 1 best threshold (max F1 on validation): 0.2556
  validation precision=0.4749 recall=0.5395 f1=0.5051


In [30]:
# Final, one-shot evaluation on the untouched TEST set -- at default 0.5 and tuned threshold.
pred1 = trainer_s1.predict(test_dataset_1)
pred_label1 = pred1.predictions.argmax(axis=1)

probs1_test = torch.softmax(torch.tensor(pred1.predictions), dim=-1)[:, 1].numpy()
pred_label1_tuned = (probs1_test >= best_threshold_s1).astype(int)

print("Stage 1 classification report @ default threshold (0.5) -- test set:")
print(classification_report(y_test, pred_label1))

print(f"\nStage 1 classification report @ tuned threshold ({best_threshold_s1:.4f}) -- test set:")
print(classification_report(y_test, pred_label1_tuned))

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Stage 1 classification report @ default threshold (0.5) -- test set:
              precision    recall  f1-score   support

           0       0.95      0.96      0.95      4797
           1       0.49      0.43      0.46       456

    accuracy                           0.91      5253
   macro avg       0.72      0.69      0.70      5253
weighted avg       0.91      0.91      0.91      5253


Stage 1 classification report @ tuned threshold (0.2556) -- test set:
              precision    recall  f1-score   support

           0       0.95      0.94      0.95      4797
           1       0.44      0.50      0.47       456

    accuracy                           0.90      5253
   macro avg       0.69      0.72      0.71      5253
weighted avg       0.91      0.90      0.90      5253



In [31]:
trainer_s1.save_model("./model_stage1_3label_optuna")
tokenizer.save_pretrained("./model_stage1_3label_optuna")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./model_stage1_3label_optuna/tokenizer_config.json',
 './model_stage1_3label_optuna/tokenizer.json')

In [32]:
from huggingface_hub import HfApi

api = HfApi()

api.create_repo(
    repo_id="evelnap/hs-stage1-3label-optuna",
    repo_type="model",
    exist_ok=True
)

api.upload_folder(
    folder_path="./model_stage1_3label_optuna",
    repo_id="evelnap/hs-stage1-3label-optuna",
    repo_type="model"
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/evelnap/hs-stage1-3label-optuna/commit/006cdeb879c4505266757260b2f75066ea43c2e1', commit_message='Upload folder using huggingface_hub', commit_description='', oid='006cdeb879c4505266757260b2f75066ea43c2e1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/evelnap/hs-stage1-3label-optuna', endpoint='https://huggingface.co', repo_type='model', repo_id='evelnap/hs-stage1-3label-optuna'), pr_revision=None, pr_num=None)

# Stage 2 — 3-Label Toxicity Type (Optuna)

In [ ]:
df_toxic = df_main[df_main["toxicity"] == 1].copy()

In [ ]:
# The 2 rare labels + minority-only-toxic rows were already dropped above, before
# Stage 1's split, so df_toxic has only the 3 kept labels.
LABEL_COLS = KEPT_LABEL_COLS

X = df_toxic["processed_text"]
y2 = df_toxic[LABEL_COLS]

In [ ]:
# Split 70/20/10. Validation drives Optuna scoring + checkpoint selection + per-label
# threshold tuning; test untouched until the end.
train_val_df_s2, test_df_s2 = train_test_split(df_toxic, test_size=0.2, random_state=42)
train_df_s2, val_df_s2 = train_test_split(train_val_df_s2, test_size=0.125, random_state=42)

train_df_s2 = train_df_s2[["processed_text"] + LABEL_COLS].copy()
val_df_s2 = val_df_s2[["processed_text"] + LABEL_COLS].copy()
test_df_s2 = test_df_s2[["processed_text"] + LABEL_COLS].copy()

print("Stage 2 split sizes:", {"train": len(train_df_s2), "val": len(val_df_s2), "test": len(test_df_s2)})

In [ ]:
# 3-label baseline: no oversampling -- Stage 2 keeps its natural label distribution.
print("Stage 2 training label counts (no oversampling):", {c: int(train_df_s2[c].sum()) for c in LABEL_COLS})

In [ ]:
train_df_s2["label"] = train_df_s2[LABEL_COLS].astype(float).values.tolist()
val_df_s2["label"] = val_df_s2[LABEL_COLS].astype(float).values.tolist()
test_df_s2["label"] = test_df_s2[LABEL_COLS].astype(float).values.tolist()

train_df_s2 = train_df_s2[["processed_text", "label"]]
val_df_s2 = val_df_s2[["processed_text", "label"]]
test_df_s2 = test_df_s2[["processed_text", "label"]]

In [ ]:
dataset_s2 = DatasetDict({
    "train": Dataset.from_pandas(train_df_s2.reset_index(drop=True)),
    "validation": Dataset.from_pandas(val_df_s2.reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df_s2.reset_index(drop=True))
})

In [ ]:
MODEL_NAME = "indolem/indobertweet-base-uncased"
tokenizer_s2 = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_s2(batch):
    return tokenizer_s2(batch["processed_text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets_2 = dataset_s2.map(tokenize_s2, batched=True)
tokenized_datasets_2 = tokenized_datasets_2.remove_columns(["processed_text"])
tokenized_datasets_2 = tokenized_datasets_2.rename_column("label", "labels")
tokenized_datasets_2.set_format("torch")

train_dataset_2 = tokenized_datasets_2["train"]
val_dataset_2 = tokenized_datasets_2["validation"]
test_dataset_2 = tokenized_datasets_2["test"]

In [ ]:
from sklearn.metrics import roc_auc_score

def compute_metrics_s2(eval_pred):
    logits, labels = eval_pred
    preds = (logits > 0).astype(int)
    return {
        "f1_micro": f1_score(labels, preds, average="micro"),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "accuracy": accuracy_score(labels, preds),
        "recall_micro": recall_score(labels, preds, average="micro", zero_division=0),
        "precision_micro": precision_score(labels, preds, average="micro", zero_division=0),
    }

## Optuna search — Stage 2

In [ ]:
def model_init_s2():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(LABEL_COLS),
        problem_type="multi_label_classification"
    )

def objective_s2(trial):
    import shutil, os
    lr = trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True)
    epochs = trial.suggest_int("num_train_epochs", 3, 6)
    batch_size = trial.suggest_categorical("per_device_train_batch_size", [8, 16])
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.1)

    args = TrainingArguments(
        output_dir=f"./optuna_s2/trial_{trial.number}",
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=16,
        learning_rate=lr,
        weight_decay=weight_decay,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        save_only_model=True,      # don't persist optimizer state during search (saves disk)
        save_total_limit=1,        # keep only the best checkpoint per trial
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        logging_steps=50,
        disable_tqdm=True,
        report_to=[],
    )

    trainer = Trainer(
        model_init=model_init_s2,
        args=args,
        train_dataset=train_dataset_2,
        eval_dataset=val_dataset_2,
        compute_metrics=compute_metrics_s2,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    try:
        trainer.train()
        metrics = trainer.evaluate()
        return metrics["eval_f1_macro"]
    finally:
        # Free disk between trials: Kaggle's working dir fills up fast, and a
        # half-written checkpoint is what triggers the torch.save iostream error.
        shutil.rmtree(f"./optuna_s2/trial_{trial.number}", ignore_errors=True)

study_s2 = optuna.create_study(direction="maximize", study_name="stage2_3label")
study_s2.optimize(objective_s2, n_trials=N_TRIALS)

print("Best Stage 2 params:", study_s2.best_params)
print("Best Stage 2 validation macro F1:", study_s2.best_value)

## Train final Stage 2 model with best hyperparameters

In [ ]:
best_s2 = study_s2.best_params

args_s2 = TrainingArguments(
    output_dir="./best_s2",
    num_train_epochs=best_s2["num_train_epochs"],
    per_device_train_batch_size=best_s2["per_device_train_batch_size"],
    per_device_eval_batch_size=16,
    learning_rate=best_s2["learning_rate"],
    weight_decay=best_s2["weight_decay"],
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    disable_tqdm=False,
    report_to=[],
)

model_s2 = model_init_s2()
trainer_s2 = Trainer(
    model=model_s2,
    args=args_s2,
    train_dataset=train_dataset_2,
    eval_dataset=val_dataset_2,
    compute_metrics=compute_metrics_s2,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
trainer_s2.train()

In [ ]:
# Tune a separate decision threshold PER LABEL on VALIDATION (never test). A single
# global 0.5 threshold is what makes rare labels collapse; per-label sweeping lets each
# label use whatever cutoff maximizes its own F1.
pred2_val = trainer_s2.predict(val_dataset_2)
probs_val = 1 / (1 + np.exp(-pred2_val.predictions))
y_val_true = pred2_val.label_ids

best_thresholds_s2 = {}
for i, label_name in enumerate(LABEL_COLS):
    p_curve, r_curve, t_curve = precision_recall_curve(y_val_true[:, i], probs_val[:, i])
    f1_curve = 2 * p_curve * r_curve / (p_curve + r_curve + 1e-12)
    best_idx = np.argmax(f1_curve[:-1])
    best_thresholds_s2[label_name] = float(t_curve[best_idx])
    print(f"{label_name}: best threshold (val)={t_curve[best_idx]:.4f} "
          f"precision={p_curve[best_idx]:.4f} recall={r_curve[best_idx]:.4f} f1={f1_curve[best_idx]:.4f}")

threshold_vector_s2 = np.array([best_thresholds_s2[c] for c in LABEL_COLS])

In [ ]:
# Final, one-shot evaluation on the untouched TEST set -- default 0.5 and per-label tuned.
pred2 = trainer_s2.predict(test_dataset_2)
probs = 1 / (1 + np.exp(-pred2.predictions))
y_true = pred2.label_ids

pred_label2 = (probs > 0.5).astype(int)
pred_label2_tuned = (probs >= threshold_vector_s2).astype(int)

print("Stage 2 classification report @ default threshold (0.5) -- test set:")
print(classification_report(y_true, pred_label2, target_names=LABEL_COLS))

print("\nStage 2 classification report @ per-label tuned thresholds -- test set:")
print(classification_report(y_true, pred_label2_tuned, target_names=LABEL_COLS))